# Group 17 - Walkthrough: Server Log Anomaly and Latency Analyzer
Run from the repo root after `python src/generate_logs.py` and `python src/parse_to_db.py`. Each step has a plain-English note.

In [ ]:
import sqlite3, numpy as np, pandas as pd
import matplotlib.pyplot as plt
con = sqlite3.connect('../data/logs.db')
df = pd.read_sql('SELECT r.minute, e.path, r.latency_ms, r.is_error FROM requests r JOIN endpoints e USING(endpoint_id)', con)
df.head()

## 1. Mean vs median vs p99
Latency is right-skewed: a few very slow requests pull the mean above the median, and p99 is far higher still.

In [ ]:
x = df.latency_ms
print(f'mean={x.mean():.1f}  median={x.median():.1f}  p95={x.quantile(.95):.1f}  p99={x.quantile(.99):.1f}')
plt.hist(x[x<800], bins=80); plt.axvline(x.mean(), color='r'); plt.axvline(x.median(), color='g'); plt.show()

## 2. Percentiles per endpoint (the SQL version is in `sql/queries.sql`, Q2)

In [ ]:
df.groupby('path').latency_ms.describe(percentiles=[.5,.95,.99]).round(1)

## 3. Lognormal check: log(latency) should look like a bell curve

In [ ]:
from scipy import stats
g = np.log(df[df.path=='/api/checkout'].latency_ms)
stats.probplot(g, dist='norm', plot=plt); plt.show()

## 4. Streaming percentile vs exact

In [ ]:
import sys; sys.path.insert(0,'../src')
from streaming import LogBucketSketch
sk = LogBucketSketch(0.01)
for v in df.latency_ms: sk.add(v)
print('exact p99', x.quantile(.99).round(1), ' sketch p99', round(sk.quantile(.99),1))

## 5. Full results
Run `python src/analysis.py` (about 30 s); it writes `reports/results.json` and the figures used in the report and website.